For new fokl_to_pyomo dev.

In [1]:
# from FoKL import FoKLRoutines
import os
dir = os.path.abspath('')  # directory of notebook
# -----------------------------------------------------------------------
# UNCOMMENT IF USING LOCAL FOKL PACKAGE:
import sys
sys.path.append(os.path.join(dir, '..', '..'))  # package directory
from src.FoKL import FoKLRoutines
from src.FoKL.fokl_to_pyomo import fokl_to_pyomo
# -----------------------------------------------------------------------
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import pandas as pd
import pyomo.environ as pyo
import pyomo.dae as dae

In [2]:
# =====================================================================
# =====================================================================
# SYSTEM-LEVEL PARAMETERS:

Tamb = 21           # ambient temperature (C)
Tmax = 75           # maximumum temperature, i.e., steady state of u=100; set s.t. u=50 yields ~55 C
t0 = 0              # start time (s)
dt = 1              # time step (s)
t_ramp = 200  # 1000       # estimated time to reach steady state (s)
t_rest = 20  # 50         # time to "rest" at steady state for sake of obtaining dT=0 training data
t_drop = 400  # 800        # estimated time to return to Tamb (s)
n = 20              # number of step tests to obtain data for

# =====================================================================
# =====================================================================
# DATA GENERATION:

u1_tests = np.linspace(100 / n, 100, n)  # heater power (%), const. values at which to obtain data
T_tests = Tamb + (Tmax - np.flip(np.logspace(np.log10(Tamb), np.log10(Tmax), n + 1))[1::])  # steady state temperature corresponding to u1_tests; decreases according to logscale as u increases

t_seg = t_ramp + t_rest + t_drop + t_rest / n  # length of single test
tf = t0 + n * t_seg
tvec = np.linspace(t0, tf, int((tf + 1) / dt))
dt = tvec[1] - tvec[0]  # redefine in case dt got rounded

u = np.zeros_like(tvec)
T = np.zeros_like(tvec) + Tamb

def T_approx(t_start, t_end, i):
    """Estimate temperature change using natural log of quadratic."""
    x = tvec[t_start:t_end] - tvec[t_start]
    H = T_tests[i] - Tamb
    W = t_end - t_start
    
    W0 = W / np.sqrt(1 - 1 / H)                 # == W'
    Q = H * (1 - ((x - W + W0) / W0 - 1) ** 2)  # == Q'
    L = np.log(Q) * H / np.log(H)               # == L'
    
    return L

t1 = int(t0)
for i in range(n):
    t2 = int(t1 + t_ramp)
    t3 = int(t1 + t_ramp + t_rest)
    t4 = int(t1 + t_ramp + t_rest + t_drop)
    
    T[t1:t2] = Tamb + T_approx(t1, t2, i)        # ramp up
    T[t2:t3] = T_tests[i]                        # steady state
    T[t3:t4] = T_tests[i] - T_approx(t3, t4, i)  # drop

    u[t1:t3] = u1_tests[i]  # const.

    t1 = int(t1 + t_seg)

T += np.random.rand(len(T))  # add noise

def u_cutoff(u):
    """Enforce [0, 100] bounds."""
    u[u < 0] = 0
    u[u > 100] = 100
    return u

u = u_cutoff(u)

# =====================================================================
# =====================================================================
# SMOOTH AND DIFFERENTIATE:

window = 9  # odd number, mean at center +/- floor(window/2))

def smooth(TS1, window):
    """Apply centered average of size window."""
    TS1_smooth = np.zeros_like(TS1)
    w2 = int(np.floor(window / 2))
    w2p1 = w2 + 1

    # bleed in:
    for i in range(w2):
        TS1_smooth[i] = np.mean(TS1[:(i + w2p1)])

    # center:
    for i in range(w2, TS1_smooth.size - w2):
        TS1_smooth[i] = np.mean(TS1[(i - w2):(i + w2p1)])

    # bleed out:
    for i in range(-w2, 0):
        TS1_smooth[i] = np.mean(TS1[(i - w2)::])

    return TS1_smooth

T_raw = T
T = smooth(T_raw, window)  # smooth

def gradient_h4(x, h):
    """h is step size. Order of error is h^4."""
    dx = np.zeros_like(x)

    # bleed in:
    h2 = 2 * h
    dx[0] = (x[1] - x[0]) / h
    dx[1] = (x[2] - x[0]) / h2

    # center difference:
    h12 = 12 * h
    for i in range(2, x.shape[0] - 2):
        dx[i] = (x[i - 2] - 8 * x[i - 1] + 8 * x[i + 1] - x[i + 2]) / h12
    
    # bleed out:
    dx[-2] = (x[-1] - x[-3]) / h2
    dx[-1] = (x[-1] - x[-2]) / h

    return dx

dT = gradient_h4(T, dt)  # derivative of smoothed
dTf = interp1d(tvec, dT, kind='previous')  # piecewise, grab previous value

# =====================================================================
# =====================================================================
# TRAIN GP:

filename = os.path.join(dir, "models", "pyomo_tclab_v10_newFunc.fokl")
try:
    GP = FoKLRoutines.load(filename)
except Exception as exception:
    GP = FoKLRoutines.FoKL(kernel=1, UserWarnings=False, aic=True)
    GP.fit([T - Tamb, u], dT, clean=True, pillow=[[0.01, 0.05], [0, 0]])
    GP.save(filename)

# Prepare Pyomo solver:
solver = pyo.SolverFactory('ipopt')
solver.options['linear_solver'] = 'ma57'

# =====================================================================
# =====================================================================
# TARGET:

# =====================================================================
# CONTROLLER REFERENCE TRAJECTORY:

# time grid
r_t0 = 0
r_tf = 999
r_dt = 1
r_n = round(r_tf / r_dt)
r_tvec = np.linspace(r_t0, r_tf, r_n + 1)

# ambient temperature
r_Tamb = 21.0

# time points of setpoint/reference values
r_t = [r_t0, 50, 150, 450, 550, r_tf]

# setpoint/reference
def r(t):
    return np.interp(t, r_t, np.array([r_Tamb, r_Tamb, 60, 60, 35, 35]) - r_Tamb)

# derivative of setpoint/reference
dr_interp = interp1d(r_t, [0, (60 - r_Tamb) / 100, 0, (35 - 60) / 100, 0, 0], kind='previous')
def dr(t):
    if isinstance(t, float) or isinstance(t, int):
        return float(dr_interp(t))
    else:  # list or ndarray
        return np.array(dr_interp(t))

# =====================================================================
# U1 BENCHMARK:

u1_benchmark = np.loadtxt(os.path.join(dir, 'data', 'u1_benchmark_solution.csv'), delimiter=',')
u1_benchmark = np.concatenate([np.array([t0, 0])[np.newaxis, :],
                               u1_benchmark[1:107, :],
                               u1_benchmark[130:218, :],
                               u1_benchmark[243:354, :],
                               u1_benchmark[370:-1, :]], axis=0)  # remove oscillations

# =====================================================================
# OPTIMIZATION USING GP MODEL:

draws_list = (np.linspace(10, 90, 9, dtype=int).tolist() +
              np.linspace(100, 1000, 10, dtype=int).tolist())
u_solution = pd.DataFrame(columns=["Time"] + draws_list)

In [3]:
# =================================================================
# PYOMO MODEL:

draws = 5

m = pyo.ConcreteModel("TCLab Heater with GP Model")
m.t = dae.ContinuousSet(bounds=(r_t0, r_tf))

m.s = pyo.Set(initialize=range(draws))  # scenarios

m.TmTamb = pyo.Var(m.t, m.s, bounds=GP.minmax[0])  # == T - Tamb
m.u = pyo.Var(m.t, bounds=(0, 100))  # same across scenarios

m.dT = dae.DerivativeVar(m.TmTamb, wrt=m.t)

for s in m.s:
    m.TmTamb[r_t0, s].fix(0.0)  # initial condition, t=0

# GP.to_pyomo([m.TmTamb, m.u], m.dT, m, m.t, m.s)
fokl_to_pyomo(GP, [m.TmTamb, m.u], m.dT, m, m.t, m.s)

/home/jacobpatrick/ESMS/dev/tclab/examples/pyomo_tclab/../../src/FoKL/fokl_to_pyomo.py:317: SyntaxWarning: Assuming 'dT' indexed by ['t', 's'].
  warnings.warn(f"Assuming '{yvar.name}' indexed by ['{t.name}', '{draws.name}'].", category=SyntaxWarning)
/home/jacobpatrick/ESMS/dev/tclab/examples/pyomo_tclab/../../src/FoKL/fokl_to_pyomo.py:324: SyntaxWarning: Assuming 'TmTamb' indexed by ['t', 's'].
  warnings.warn(f"Assuming '{xvar.name}' indexed by ['{t.name}', '{draws.name}'].", category=SyntaxWarning)
/home/jacobpatrick/ESMS/dev/tclab/examples/pyomo_tclab/../../src/FoKL/fokl_to_pyomo.py:332: SyntaxWarning: Assuming 'u' indexed by 't'.
  warnings.warn(f"Assuming '{xvar.name}' indexed by '{t.name}'.", category=SyntaxWarning)


In [4]:
m.pprint()

4 Set Declarations
    GP0_attributes : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {0, 1}
    GP0_orders : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    6 : {1, 2, 3, 4, 5, 6}
    GP0_terms : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :   28 : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27}
    s : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {0, 1, 2, 3, 4}

1 Param Declarations
    GP0_beta : Size=140, Index=s*GP0_terms, Domain=Any, Default=None, Mutable=True
        Key     : Value
         (0, 0) :  0.39295686235944466
         (0, 1) :    1.271129148200484
         (0, 2) :  -0.8835091572388022
         (0, 3) :  -1.7263093911